# untell — Train the GPU Moat (GRPO + LoRA)

This notebook trains a small instruct model (Qwen2.5-3B) to humanize AI text in a *single forward pass* — no inference loop, no API key. The model learns to evade the full local detector ensemble via GRPO (group-relative policy optimization).

**Free GPU:** This runs on a single T4 (16GB) on Kaggle / Colab. **Watch the step time and Kaggle's 9h GPU limit:** each step is ~100s, so 300 steps = 8.3h max. Defaults are tuned for this limit.

## What you'll get
- A LoRA adapter that turns Qwen2.5-3B into a single-pass humanizer
- The policy works at inference with: `untell-eval-policy --policy out/rl-humanizer`
- Or via the local rewriter: `UNTELL_POLICY_DIR=out/rl-humanizer untell humanize "text"`

In [ ]:
# Install untell with training deps
!pip install -q "untell[train,full]"
!pip install -q "huggingface_hub"

# Fix torchao/peft version conflict on Kaggle
!pip install -q "torchao>=0.16.0" 2>/dev/null || pip install -q "peft<0.14"

# Check GPU
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Option A: Train against the local ensemble (free, ~2h)

This trains the policy to beat our 7 local detectors. The resulting model will be strong against open-source detectors. To also beat commercial detectors (GPTZero, Originality), see Option B below.

In [ ]:
# Smoke test: verify the pipeline works end-to-end with a tiny model
!python -m training.rl_humanizer --smoke
print("Smoke test passed — pipeline is ready for full training.")

In [ ]:
# Full training: Qwen2.5-3B-Instruct, GRPO + LoRA, 500 steps, full detector tier
# This runs for ~2-4 hours on a T4 GPU.
# Set your HF token to push the adapter so it survives the ephemeral GPU host:
# import os; os.environ["HF_TOKEN"] = "your_hf_token_here"

!python -m training.rl_humanizer \
    --model Qwen/Qwen2.5-3B-Instruct \
    --tier full \
    --steps 500 \
    --k 6 \
    --out out/rl-humanizer \
    --load-4bit \
    --hub-id "your-username/untell-rl-humanizer"  # optional: push to HF

print("Training complete!")

## Option B: Train against a GPTZero surrogate (stronger, needs API labels)

The local ensemble does NOT transfer to commercial detectors. To target GPTZero specifically:
1. Collect GPTZero API scores for ~2000 AI/human text pairs
2. Train a surrogate model that mimics GPTZero
3. Train the policy against the surrogate

This is a paid path (GPTZero API credits) but produces a stronger policy.

In [ ]:
# Step 1: Train a surrogate of GPTZero from your labeled CSV
# Format: text,score (score = GPTZero's P(AI) in [0,1])
# !python -m training.surrogate --dataset gptzero_labels.csv --out out/surrogate

# Step 2: Train the policy against the surrogate
# import os; os.environ["UNTELL_SURROGATE_DIR"] = "out/surrogate"
# !python -m training.rl_humanizer ... (same as Option A)

## Evaluate the trained policy

After training, evaluate against held-out text:

In [ ]:
# A/B eval: trained policy vs untuned base model
!untell-eval-policy \
    --policy out/rl-humanizer \
    --n 50 \
    --tier full \
    --vs-base \
    --json

## Use the trained policy

On any machine with the adapter downloaded:

In [ ]:
# Set the policy path
import os
os.environ["UNTELL_POLICY_DIR"] = "out/rl-humanizer"

# Now the loop auto-selects the policy as the rewriter
from untell.rewriter import get_rewriter
rw = get_rewriter()
print(f"Active rewriter: {rw.name if rw else 'None'}")

# Humanize a sample
text = "Furthermore, artificial intelligence has fundamentally transformed numerous industries. Moreover, organizations leverage these technologies to optimize efficiency."
from untell.scripts.run import untell_text
result = untell_text(text, tier="full", rewriter=rw)
print(f"Before: P(AI)={result['pre']['max']:.2f}")
print(f"After:  P(AI)={result['post']['max']:.2f}")
print(f"Output: {result['final'][:200]}...")
print(f"Iterations: {result['iterations']}")

## Upload adapter to HuggingFace Hub

If you didn't use `--hub-id` during training, you can upload manually:

In [ ]:
from huggingface_hub import HfApi, notebook_login
notebook_login()  # Log in with your HF token
api = HfApi()
api.create_repo("your-username/untell-rl-humanizer", repo_type="model", exist_ok=True)
api.upload_folder(
    folder_path="out/rl-humanizer",
    repo_id="your-username/untell-rl-humanizer",
    repo_type="model",
)
print("Uploaded!")